# Hyper Parameter Tuning
- 파라미터: 모델이 데이터 학습을 통해 스스로 찾아내는 값(예: 회귀 계수)
- 하이퍼 파라미터: 사용자가 직접 모델에 설정해주는 값(예: KNN의 n_neighbors)

## GridSearchCV (격자 탐색)
- 방식: 가능한 모든 조합을 미리 격자처럼 정해놓고 하나씩 다 대입해보는 방식
- 장점: 모든 경우의 수를 따지므로 '전역 최적점'을 찾을 확률이 높음
- 단점: 파라미터 개수가 많아질수록 실행 시간이 기하급수적으로 늘어남

In [32]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [33]:
fish_df = pd.read_csv('data/fish.csv')
fish_df

,Species,Weight,Length,Diagonal,Height,Width
0,Bream,242.0,25.4,30.0,11.5200,4.0200
1,Bream,290.0,26.3,31.2,12.4800,4.3056
2,Bream,340.0,26.5,31.1,12.3778,4.6961
3,Bream,363.0,29.0,33.5,12.7300,4.4555
4,Bream,430.0,29.0,34.0,12.4440,5.1340
...,...,...,...,...,...,...
154,Smelt,12.2,12.2,13.4,2.0904,1.3936
155,Smelt,13.4,12.4,13.5,2.4300,1.2690
156,Smelt,12.2,13.0,13.8,2.2770,1.2558
157,Smelt,19.7,14.3,15.2,2.8728,2.0672


In [34]:
# target 데이터 인코딩
from sklearn.preprocessing import LabelEncoder, StandardScaler

encoder = LabelEncoder()
fish_df['Target'] = encoder.fit_transform(fish_df['Species'])

print(encoder.classes_)
fish_df

['Bream' 'Parkki' 'Perch' 'Pike' 'Roach' 'Smelt' 'Whitefish']


,Species,Weight,Length,Diagonal,Height,Width,Target
0,Bream,242.0,25.4,30.0,11.5200,4.0200,0
1,Bream,290.0,26.3,31.2,12.4800,4.3056,0
2,Bream,340.0,26.5,31.1,12.3778,4.6961,0
3,Bream,363.0,29.0,33.5,12.7300,4.4555,0
4,Bream,430.0,29.0,34.0,12.4440,5.1340,0
...,...,...,...,...,...,...,...
154,Smelt,12.2,12.2,13.4,2.0904,1.3936,5
155,Smelt,13.4,12.4,13.5,2.4300,1.2690,5
156,Smelt,12.2,13.0,13.8,2.2770,1.2558,5
157,Smelt,19.7,14.3,15.2,2.8728,2.0672,5


In [35]:
# 데이터 분할
from sklearn.model_selection import train_test_split

X = fish_df.drop(['Species', 'Target'], axis=1).to_numpy()
y = fish_df['Target'].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

# KNN은 거리기반 연산이므로 스케일링 필요
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [36]:
# 최근접 이웃 모델 - 최적의 n_neighbors 찾기
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier()

# 탐색할 파라미터 설정 
params = {
    # 1에서 30까지 모두 대입해보기
    'n_neighbors' : range(1, 31)
}

grid_search_knn = GridSearchCV(knn, param_grid=params, cv=5, scoring='accuracy')
grid_search_knn.fit(X_train_scaled, y_train)

print("KNN 최적 파라미터: ", grid_search_knn.best_params_)
print("KNN 최고 검증 정확도: ", grid_search_knn.best_score_)

KNN 최적 파라미터:  {'n_neighbors': 3}
KNN 최고 검증 정확도:  0.8193846153846154


In [37]:
# LogisticRegression 분류 모델
from sklearn.linear_model import LogisticRegression

lr_clf = LogisticRegression()

# 여러 파라미터를 조합하여 최상의 조합 찾기
params = {
    'max_iter': [1000, 2000, 3000],      # 모델이 정답을 찾기 위해 몇번 반복할지
    'C': [0.01, 0.1, 1, 10, 100]         # 규제 강도: 모델이 너무 복잡해지는 것을 방지하기 위한 패널티 수치
}

# refit=True : 가장 좋은 설정값을 찾으면 그 설정으로 전체 데이터를 다시 학습시켜 최종 모델을 만든다.
grid_search_lr = GridSearchCV(lr_clf, param_grid=params, cv=5, scoring='accuracy', refit=True)
grid_search_lr.fit(X_train_scaled, y_train)

# 결과 리포트
print(f"최고의 검증 정확도 : {grid_search_lr.best_score_}")
print(f"최적의 파라미터 : {grid_search_lr.best_params_}")

# 최종 완성 된 모델 추출
best_lr_model = grid_search_lr.best_estimator_
print(f"테스트 데이터 최종 성능: {best_lr_model.score(X_test_scaled, y_test)}")

최고의 검증 정확도 : 0.9286153846153846
최적의 파라미터 : {'C': 100, 'max_iter': 1000}
테스트 데이터 최종 성능: 0.875


In [38]:
# 딕셔너리 형태의 결과를 데이터 프레임으로 변환해서 확인
cv_results_df = pd.DataFrame(grid_search_lr.cv_results_)

# 주요 항목만 골라 보기 (성능 순위대로 정렬)
display_cols = ['params', 'mean_test_score', 'rank_test_score', 'mean_fit_time']
cv_results_df[display_cols].sort_values(by='rank_test_score')

,params,mean_test_score,rank_test_score,mean_fit_time
14,"{'C': 100, 'max_iter': 3000}",0.928615,1,0.012976
12,"{'C': 100, 'max_iter': 1000}",0.928615,1,0.013721
13,"{'C': 100, 'max_iter': 2000}",0.928615,1,0.013547
9,"{'C': 10, 'max_iter': 1000}",0.873846,4,0.010168
11,"{'C': 10, 'max_iter': 3000}",0.873846,4,0.008698
10,"{'C': 10, 'max_iter': 2000}",0.873846,4,0.008867
6,"{'C': 1, 'max_iter': 1000}",0.819385,7,0.007885
7,"{'C': 1, 'max_iter': 2000}",0.819385,7,0.005556
8,"{'C': 1, 'max_iter': 3000}",0.819385,7,0.006155
3,"{'C': 0.1, 'max_iter': 1000}",0.716923,10,0.003616


## RandomizedSearchCV
모든 조합을 다 시도하는 대신, 정해진 횟수만큼 무작위로 샘플링하여 검증한다.
- 파라미터 범위가 매우 넓거나 종류가 많을 때, 적은 연산으로 충분히 좋은 성능을 낼 수 있다.

In [39]:
from sklearn.datasets import load_wine

wine = load_wine()
wine_df = pd.DataFrame(wine.data, columns=wine.feature_names)
wine_df['target'] = wine.target
wine_df

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,13.71,5.65,2.45,20.5,95.0,1.68,0.61,0.52,1.06,7.70,0.64,1.74,740.0,2
174,13.40,3.91,2.48,23.0,102.0,1.80,0.75,0.43,1.41,7.30,0.70,1.56,750.0,2
175,13.27,4.28,2.26,20.0,120.0,1.59,0.69,0.43,1.35,10.20,0.59,1.56,835.0,2
176,13.17,2.59,2.37,20.0,120.0,1.65,0.68,0.53,1.46,9.30,0.60,1.62,840.0,2


In [40]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
import time

# 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(
    wine.data, wine.target, test_size=0.2, random_state=42, stratify=wine.target
    )

# 스케일링
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 공통 하이퍼파라미터 그리드 설정
# 전체 조합의 개수 2 * 4 * 3 = 24 
params = {
    'max_iter': [3000, 5000],      
    'C': [0.1, 1, 10, 100],
    'solver': ['lbfgs', 'saga', 'newton-cg']         
}

classifier = LogisticRegression()

# GridSearchCV -> 모든 조합을 전수 조사
start = time.time()
grid = GridSearchCV(classifier, param_grid=params, cv=5, scoring='accuracy')
grid.fit(X_train_scaled, y_train)
grid_time = time.time() - start

# RandomizedSearchCV -> 무작위로 n_iter 개수만큼 조사
start = time.time()
rand = RandomizedSearchCV(
    classifier, params, n_iter=10, cv=5, scoring='accuracy', random_state=42
    )
rand.fit(X_train_scaled, y_train)
rand_time = time.time() - start

print("그리드서치 최고 검증 점수 : ", grid.best_score_)
print("그리드서치 최적 파라미터 : ", grid.best_params_)
print("그리드서치 테스트 점수 : ", grid.score(X_test_scaled, y_test))

print("랜덤서치 최고 검증 점수 : ", rand.best_score_)
print("랜덤서치 최적 파라미터 : ", rand.best_params_)
print("랜덤서치 테스트 점수 : ", rand.score(X_test_scaled, y_test))

time_diff = grid_time / rand_time
print(f"랜덤 서치가 그리드 서치보다 약 {time_diff}배 빨랐습니다.")



그리드서치 최고 검증 점수 :  0.9862068965517242
그리드서치 최적 파라미터 :  {'C': 0.1, 'max_iter': 3000, 'solver': 'lbfgs'}
그리드서치 테스트 점수 :  1.0
랜덤서치 최고 검증 점수 :  0.9862068965517242
랜덤서치 최적 파라미터 :  {'solver': 'lbfgs', 'max_iter': 3000, 'C': 0.1}
랜덤서치 테스트 점수 :  1.0
랜덤 서치가 그리드 서치보다 약 2.2604102454075155배 빨랐습니다.
